# Phase 1 — Extract Atomic Statements & Collect Critiques

**Flow:**
1. Load opinions → extract atomic statements → compress duplicates
2. Review the compressed list — **STOP** to collect additions/critiques
3. Finalise the statement list → save for Pol.is-style voting in `02_analyse.ipynb`
4. *(Optional)* After voting, synthesise a group statement from the winners

The statements produced here are short and single-idea — suited to Pol.is voting.
The comprehensive group statement comes *after* voting, built from the winners.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
TOPIC        = "Where should we hold the combined summer social for engineering and marketing?"
OPINIONS_CSV = "example_opinions.csv"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os, json, asyncio
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import pandas as pd
from group_consensus.models.types import Opinion, Participant, SessionConfig, StatementType
from group_consensus.mediation.async_atomic_model import AsyncAtomicStatementModel

OUTPUT_DIR = Path("session_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"API key : {'✓' if os.getenv('ANTHROPIC_API_KEY') else '⚠  not found'}")

---
## Step 1 — Load opinions

In [ ]:
df = pd.read_csv(OPINIONS_CSV)
print(f"{len(df)} responses — columns: {list(df.columns)}")
df

In [ ]:
NAME_COL    = df.columns[1]   # adjust index if needed
OPINION_COL = df.columns[2]
SESSION_ID  = "session_01"

participants = []
opinions     = []
for i, row in df.iterrows():
    pid  = f"p{i}"
    name = str(row[NAME_COL]).strip()
    text = str(row[OPINION_COL]).strip()
    participants.append(Participant(id=pid, name=name))
    opinions.append(Opinion(participant_id=pid, text=text, session_id=SESSION_ID))

name_to_pid = {p.name: p.id for p in participants}

config = SessionConfig(session_id=SESSION_ID, topic=TOPIC)
model  = AsyncAtomicStatementModel(config)

print(f"✓ {len(participants)} participants")

---
## Step 2 — Extract atomic statements

The LLM reads all opinions and pulls out one-idea statements (5–15 words each).
Expect 15–30 raw statements before compression.

In [ ]:
print("Extracting atomic statements from opinions...")
raw_statements = asyncio.run(
    model.extract(topic=TOPIC, opinions=opinions, participants=participants)
)
print(f"\n{len(raw_statements)} raw statements extracted:\n")
for i, s in enumerate(raw_statements, 1):
    print(f"  {i:>2}. {s.text}")

---
## Step 3 — Compress: merge equivalents, flag contradictions

The LLM merges semantic duplicates ("loud" / "noisy") and compatible constraints
("by 8pm" subsumes "by 9pm" — the 9pm person is satisfied by it).

Genuine contradictions are kept as separate **CONTESTED** statements — they stay
in the voting pool so Pol.is can show which groups hold which position.

In [ ]:
print("Compressing...")
compressed = asyncio.run(
    model.compress(topic=TOPIC, statements=raw_statements)
)

atomic     = [s for s in compressed.statements if s.type == StatementType.ATOMIC]
contested  = [s for s in compressed.statements if s.type == StatementType.CONTESTED]

print(f"\n{len(compressed.statements)} statements after compression  "
      f"({len(raw_statements) - len(compressed.statements)} merged)")
print(f"  {len(atomic)} agreed atomic  |  {len(contested)} contested\n")

print("ATOMIC STATEMENTS (clear single ideas):")
for i, s in enumerate(atomic, 1):
    print(f"  {i:>2}. {s.text}")

if contested:
    print("\nCONTESTED STATEMENTS (genuine disagreements — kept for voting):")
    for i, s in enumerate(contested, 1):
        print(f"  {i:>2}. {s.text}")

In [ ]:
# Show what was merged and why
if compressed.merges:
    print(f"MERGES ({len(compressed.merges)}):")
    for m in compressed.merges:
        print(f"  Kept    : {m.kept}")
        for a in m.absorbed:
            print(f"  Dropped : {a}")
        print(f"  Reason  : {m.reason}\n")

if compressed.contested_pairs:
    print(f"CONTESTED PAIRS ({len(compressed.contested_pairs)}):")
    for a, b in compressed.contested_pairs:
        print(f"  ↔  '{a}'")
        print(f"     '{b}'\n")

---
## ⏸  STOP HERE — review and add anything missing

Look through the statement list. If anything important from the original opinions
didn't make it through, add it in the cell below.

You can also share the list with participants and ask:
> *"Is anything missing from this list that matters to you?"*

Add any additions as short declarative statements (5–15 words).
Leave the list empty if nothing needs adding.

---

In [ ]:
from group_consensus.models.types import Statement, StatementType

# ── ADD ANY MISSING STATEMENTS HERE ──────────────────────────────────────────
manual_additions = [
    # "Non-alcoholic drink options are available",
    # "The venue has good natural light",
]
# ─────────────────────────────────────────────────────────────────────────────

additional_stmts = [
    Statement(text=t, type=StatementType.ATOMIC, session_id=SESSION_ID)
    for t in manual_additions
]

final_statements = compressed.statements + additional_stmts

print(f"Final statement list: {len(final_statements)} statements")
print(f"  ({len(compressed.statements)} from extraction + {len(additional_stmts)} manual)\n")
for i, s in enumerate(final_statements, 1):
    tag = " [CONTESTED]" if s.type == StatementType.CONTESTED else ""
    print(f"  {i:>2}. {s.text}{tag}")

---
## Step 4 — Save and generate voting form template

In [ ]:
# Save for 02_analyse.ipynb
session_data = {
    "session_id" : SESSION_ID,
    "topic"      : TOPIC,
    "statements" : [
        {"id": str(s.id), "text": s.text, "type": s.type}
        for s in final_statements
    ],
}
with open(OUTPUT_DIR / "statements.json", "w") as f:
    json.dump(session_data, f, indent=2)

label_map = {
    f"Statement {i}": {"id": str(s.id), "text": s.text, "type": s.type}
    for i, s in enumerate(final_statements, 1)
}
with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print("✓ Saved statements.json and label_map.json\n")

# Voting form template
DIVIDER = "═" * 60
print(DIVIDER)
print("  VOTING FORM — copy into a shared spreadsheet or form")
print(DIVIDER)
print(f"\nTopic: {TOPIC}")
print("One multiple-choice question per statement (Agree / Pass / Disagree):\n")
for label, info in label_map.items():
    tag = " [CONTESTED]" if info["type"] == StatementType.CONTESTED else ""
    print(f"  {label}{tag}")
    print(f"  {info['text']}\n")

---
## Step 5 — (Optional) Synthesise group statement after voting

Run this section **after** completing `02_analyse.ipynb` and identifying
the bridging statements. Paste those statement labels below.

This produces the comprehensive group statement from the atomic winners.

In [ ]:
# ── PASTE BRIDGING STATEMENT LABELS FROM 02_analyse.ipynb ────────────────────
# e.g. ["Statement 1", "Statement 3", "Statement 5"]
BRIDGING_LABELS = []
# ─────────────────────────────────────────────────────────────────────────────

if not BRIDGING_LABELS:
    print("Fill in BRIDGING_LABELS after running 02_analyse.ipynb.")
else:
    with open(OUTPUT_DIR / "label_map.json") as f:
        label_map = json.load(f)

    from uuid import UUID
    bridging_stmts = [
        Statement(
            id=UUID(label_map[lbl]["id"]),
            text=label_map[lbl]["text"],
            type=StatementType.ATOMIC,
            session_id=SESSION_ID,
        )
        for lbl in BRIDGING_LABELS
        if lbl in label_map
    ]

    print(f"Synthesising group statement from {len(bridging_stmts)} bridging statements...\n")
    group_statement = asyncio.run(
        model.synthesise(topic=TOPIC, winning_statements=bridging_stmts)
    )

    print("═" * 60)
    print("  GROUP CONSENSUS STATEMENT")
    print("═" * 60)
    print()
    print(group_statement)
    print()

    with open(OUTPUT_DIR / "group_statement.txt", "w") as f:
        f.write(group_statement)
    print("✓ Saved to session_data/group_statement.txt")